### Import

In [12]:
import pandas as pd
import mlflow

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import Ridge

from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, root_mean_squared_log_error

In [2]:
df = pd.read_csv("../data/processed/rossmannV2.csv")

In [3]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("rossmann-forecasting")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1787652399876, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1787652399876, lifecycle_stage='active', name='rossmann-forecasting', tags={}, trace_location=None, workspace='default'>

---

### Splitting the data and Metric choosing

In [4]:
df = df.sort_values("Date").reset_index(drop=True)

cutoff = "2015-01-01"

train = df[df["Date"] < cutoff].copy()
test = df[df["Date"] >= cutoff].copy()

In [5]:
X_train = train.drop(columns=["Sales", "Date"])
y_train = train["Sales"]

X_test = test.drop(columns=["Sales", "Date"])
y_test = test["Sales"]

In [6]:
tscv = TimeSeriesSplit(
    n_splits=5,
    gap=7
)

Here i added gap for so i have 7 rows gap between splits but it'll split wrong several times since dataset has several stores.

In [8]:
for train_idx, valid_idx in tscv.split(X_train):
    X_fold_train = X_train.iloc[train_idx]
    X_fold_valid = X_train.iloc[valid_idx]

    y_fold_train = y_train.iloc[train_idx]
    y_fold_valid = y_train.iloc[valid_idx]

Prevents future data leaking.

---

Also i decided to use RMSLE for final model since it penalizes relative errors and is appropriate for sales with large variation.

But for model selection I'll use MAE, RMSE and RMSLE.

---

### Baseline and Model choosing

In [9]:
ridge_pipeline = Pipeline([
    ("scaler", RobustScaler()),
    ("model", Ridge(alpha=1.0))
])

In [13]:
ridge_pipeline.fit(X_fold_train, y_fold_train)

ridge_predict = ridge_pipeline.predict(X_fold_valid)

result_ridge = root_mean_squared_log_error(y_fold_valid, ridge_predict)

print(result_ridge)

ValueError: Root Mean Squared Logarithmic Error cannot be used when targets contain values less than or equal to -1.

In [14]:
print("y_test min:", y_fold_valid.min())
print("y_pred min:", ridge_predict.min())

print("Negative y_test:", (y_fold_valid < 0).sum())
print("Negative y_pred:", (ridge_predict < 0).sum())

y_test min: 0
y_pred min: -1873.7518547727173
Negative y_test: 0
Negative y_pred: 9598
